# Demo 03 — data quality contract и Parquet reconciliation

Цель: показать, что успешный `show()` не доказывает корректность pipeline. Проверяем quality gates, агрегаты до/после записи и идемпотентный output.

In [ ]:
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

from pyspark.sql import functions as F
from mentor_spark_lab.notebook_support import create_spark_session
from mentor_spark_lab.pipeline import MarketplacePipeline, ensure_output_parent
from mentor_spark_lab.schemas import CUSTOMER_SCHEMA, EVENT_SCHEMA

spark = create_spark_session("lesson04-notebook-quality-roundtrip")
spark.sparkContext.setLogLevel("WARN")
pipeline = MarketplacePipeline(spark)

## 1. Детерминированные дефекты

Во входе есть валидные покупки, отрицательная сумма, `null amount` и событие другого типа. Explicit schema фиксирует типы независимо от содержимого sample.

In [ ]:
events = spark.createDataFrame(
    [
        (1, datetime(2025, 8, 1, 10, 0), 101, "purchase", Decimal("100.00"), "web"),
        (2, datetime(2025, 8, 1, 11, 0), 102, "purchase", Decimal("50.00"), "ios"),
        (3, datetime(2025, 8, 1, 12, 0), 101, "purchase", Decimal("-5.00"), "web"),
        (4, datetime(2025, 8, 1, 13, 0), 102, "purchase", None, "android"),
        (5, datetime(2025, 8, 1, 14, 0), 101, "view", Decimal("10.00"), "web"),
    ],
    EVENT_SCHEMA,
)
customers = spark.createDataFrame(
    [
        (101, "RU", "new", date(2025, 1, 1)),
        (102, "KZ", "loyal", date(2024, 1, 1)),
    ],
    CUSTOMER_SCHEMA,
)
events.printSchema()
events.show(truncate=False)

In [ ]:
quality = events.agg(
    F.count("*").alias("input_rows"),
    F.sum(F.when(F.col("amount").isNull(), 1).otherwise(0)).alias("null_amount"),
    F.sum(F.when(F.col("amount") <= 0, 1).otherwise(0)).alias("non_positive_amount"),
).first()
assert quality.input_rows == 5
assert quality.null_amount == 1
assert quality.non_positive_amount == 1
quality

## 2. Business-valid mart

Quality gate должен оставить две покупки. Затем enrichment и агрегация дают две строки mart.

In [ ]:
purchases = pipeline.valid_purchases(events)
result = pipeline.daily_revenue(purchases, customers)
metrics = pipeline.collect_metrics(events, purchases, result)
assert metrics.input_events == 5
assert metrics.valid_purchases == 2
assert metrics.output_rows == 2
assert metrics.revenue_total == Decimal("150.00")
result.show(truncate=False)
metrics

## 3. Idempotent Parquet + round-trip

`overwrite` делает повторный запуск воспроизводимым. После записи перечитываем Parquet и сверяем row count и revenue, а не доверяем отсутствию exception.

In [ ]:
output_root = "/workspace/labs/spark/data/output/notebook-quality"
ensure_output_parent(output_root)
pipeline.write_output(result, output_root)
assert pipeline.validate_roundtrip(output_root, metrics)
parquet_files = sorted(Path(output_root).rglob("*.parquet"))
assert parquet_files
print(f"PASS output_roundtrip: rows={metrics.output_rows}, revenue={metrics.revenue_total}")
print(f"physical parquet files={len(parquet_files)}")
[str(path.relative_to(output_root)) for path in parquet_files]

### Самостоятельный эксперимент

Убери `partitionBy("event_date")`, затем добавь второй день. Сравни layout директорий и объясни, почему слишком высокая cardinality partition key создаёт small-files problem.

In [ ]:
spark.stop()
print("PASS quality_and_parquet_demo")